# Stable Diffusion XL — Fallback Image Generator
Upload your `manifest_*.json` file to this Colab notebook to generate any missing images using a massive GPU!

**Instructions:**
1. Click the Folder icon on the left sidebar.
2. Drag and drop your `manifest_xxx.json` file there.
3. Run all the cells below in order.

In [ ]:
# 1. Install Dependencies
!pip install -q diffusers>=0.25.0 transformers>=4.36.0 accelerate>=0.25.0 safetensors>=0.4.0 invisible_watermark>=0.2.0 tqdm>=4.66.0

In [ ]:
# 2. Setup Directories & Load Manifest
import os
import json
from pathlib import Path
import glob

manifest_files = glob.glob("/content/manifest_*.json")
if not manifest_files:
    raise FileNotFoundError("Please upload your manifest.json file to the /content directory first!")

manifest_path = manifest_files[0]
print(f"Found manifest: {manifest_path}")

with open(manifest_path, "r") as f:
    manifest = json.load(f)

IMAGES_DIR = Path("/content/images")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

missing_images = []
for i, block in enumerate(manifest.get("blocks", [])):
    if block.get("clip_path") is None and block.get("fallback_prompt"):
        segment_id = block.get("segment_id", f"segment_{i+1:03d}")
        img_name = f"{segment_id}.jpg"
        out_path = IMAGES_DIR / img_name
        missing_images.append({"prompt": block["fallback_prompt"], "output_path": out_path})

print(f"Found {len(missing_images)} scenes that need AI images.")

In [ ]:
# 3. Load Stable Diffusion XL Model
import torch
from diffusers import StableDiffusionXLPipeline

if len(missing_images) > 0:
    print("Loading SDXL base model...")
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0", 
        torch_dtype=torch.float16, 
        variant="fp16", 
        use_safetensors=True
    )
    pipe = pipe.to("cuda")
    print("Model loaded successfully!")
else:
    print("No images needed. You can skip this cell.")

In [ ]:
# 4. Generate Images
from tqdm import tqdm

if len(missing_images) > 0:
    print("Generating Images...")
    for task in tqdm(missing_images):
        prompt = task["prompt"] + ", high quality, 8k resolution, cinematic lighting, masterpiece"
        negative_prompt = "blurry, low quality, watermark, text, signature, bad anatomy"
        image = pipe(prompt=prompt, negative_prompt=negative_prompt, num_inference_steps=30, width=1080, height=1920).images[0]
        image.save(task["output_path"])
        print(f"Saved: {task['output_path']}")
        
    print("\nAll images generated successfully!")

In [ ]:
# 5. Zip and Download
from google.colab import files
import shutil

if len(missing_images) > 0:
    print("Zipping images...")
    shutil.make_archive("/content/images", "zip", "/content/images")
    print("Downloading images.zip to your computer...")
    files.download("/content/images.zip")
else:
    print("Nothing to download.")